# Pipeline ETL — Tweets y Análisis de Sentimiento
**Proyecto:** food-price-forecasting-peru | Big Data Analytics UNMSM 2025-II

**Descripción:** Limpieza, normalización y análisis de sentimiento de tweets sobre precios de alimentos en Lima.
Produce dos salidas en Zona Plata:
- **tweets_sentimiento**: un registro por tweet con score y clasificación
- **tweets_sentimiento_diario**: métricas agregadas por día

**Flujo:**
1. Cargar JSON desde GCS (Zona Bronce)
2. Eliminar duplicados por ID
3. Convertir fecha a datetime
4. Limpiar y normalizar texto
5. Imputar nulos en likes y retweets
6. Aplicar BERT (pysentimiento) tweet por tweet
7. Crear variable de **día**
8. Agregar por **día** (métricas para el modelo predictivo, facilitando el join con datos de precios)
9. Guardar ambas tablas en Zona Plata como Parquet

In [6]:
# Instalación de dependencias
# Ejecutar solo una vez por sesión de Colab
!pip install -q pyspark google-cloud-storage pysentimiento

In [7]:
BUCKET_NAME  = "food-price-peru-2025-01"
BRONCE_PATH  = f"gs://{BUCKET_NAME}/bronce/tweets/bronce_tweets_precios_lima.json"
PLATA_TWEETS = f"gs://{BUCKET_NAME}/plata/tweets_sentimiento"
PLATA_SEMANAL = f"gs://{BUCKET_NAME}/plata/tweets_sentimiento_semanal"

print(f"Bucket : {BUCKET_NAME}")
print(f"Entrada: {BRONCE_PATH}")
print(f"Salida tweets : {PLATA_TWEETS}")
print(f"Salida semanal : {PLATA_SEMANAL}")

Bucket : food-price-peru-2025-01
Entrada: gs://food-price-peru-2025-01/bronce/tweets/bronce_tweets_precios_lima.json
Salida tweets : gs://food-price-peru-2025-01/plata/tweets_sentimiento
Salida semanal : gs://food-price-peru-2025-01/plata/tweets_sentimiento_semanal


In [8]:
# Autenticación GCP y configuración de Spark con conector GCS
import os, re
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, StringType, StructType, StructField, LongType
from google.colab import auth

auth.authenticate_user()

# Descargar el conector GCS para Spark si no existe en la sesión
jar_name = "gcs-connector.jar"
jar_path = os.path.abspath(jar_name)
if not os.path.exists(jar_path):
    !wget -q https://repo1.maven.org/maven2/com/google/cloud/bigdataoss/gcs-connector/hadoop3-2.2.14/gcs-connector-hadoop3-2.2.14-shaded.jar -O {jar_name}

os.environ["PYSPARK_SUBMIT_ARGS"] = f"--jars {jar_path} pyspark-shell"

spark = (SparkSession.builder
    .appName("ETL_Tweets_Sentimiento")
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.driver.extraClassPath", jar_path)
    .config("spark.executor.extraClassPath", jar_path)
    # Aumentar memoria del driver para cargar el modelo BERT
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "4")   # Dataset pequeño, no necesita más
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark iniciado correctamente. Versión:", spark.version)

Spark iniciado correctamente. Versión: 4.0.3


In [9]:
# Carga desde GCS y verificación inicial

df_raw = spark.read.option("multiLine", "true").json(BRONCE_PATH)

print("─" * 50)
print(f"Registros cargados : {df_raw.count()}")
print(f"Columnas           : {df_raw.columns}")
print("─" * 50)
print("Schema inferido por Spark:")
df_raw.printSchema()

# Vista previa de los primeros 3 registros
df_raw.limit(3).toPandas()

──────────────────────────────────────────────────
Registros cargados : 5133
Columnas           : ['comentario', 'distrito', 'fecha', 'id', 'likes', 'retweets', 'termino_busqueda']
──────────────────────────────────────────────────
Schema inferido por Spark:
root
 |-- comentario: string (nullable = true)
 |-- distrito: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- id: long (nullable = true)
 |-- likes: long (nullable = true)
 |-- retweets: long (nullable = true)
 |-- termino_busqueda: string (nullable = true)



,comentario,distrito,fecha,id,likes,retweets,termino_busqueda
0,SUBE PRECIO DE PAPA a S/.2 y 4 EN MERCADO DE L...,None,2015-10-15T04:22:46+00:00,654512717814960128,1,0,precio papa Lima
1,"Precio de papa se dispara en Lima, es el resul...",None,2016-10-04T17:41:15+00:00,783361353436332033,5,15,precio papa Lima
2,"Precio mayorista de papa, pollo, manzana y uva...",None,2016-05-14T02:07:05+00:00,731304797631053825,11,11,precio papa Lima


In [10]:
# ETL: limpieza y transformaciones
# ─────────────────────────────────────────────────────────────────────
# Transformaciones en orden:
# 1. dropDuplicates por id — evita que un tweet influya más de una vez
# 2. Conversión de fecha a timestamp con timezone UTC
# 3. Imputación de nulos en likes y retweets con 0
#    (177 nulos en likes y 714 en retweets detectados en EDA)
# 4. Limpieza de texto sobre la columna "comentario":
#    - Convertir a minúsculas
#    - Eliminar URLs (http/https)
#    - Eliminar menciones (@usuario)
#    - Eliminar el símbolo # pero conservar la palabra del hashtag
#      porque contiene información semántica relevante
#    - Eliminar caracteres no alfabéticos ni espacios
#    - Colapsar espacios múltiples
# ─────────────────────────────────────────────────────────────────────

def clean_text(text: str) -> str:
    """Limpia y normaliza el texto de un tweet."""
    if not text:
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)        # eliminar URLs
    text = re.sub(r"@\w+", "", text)                   # eliminar menciones
    text = re.sub(r"#", "", text)                       # eliminar símbolo # pero conservar palabra
    text = re.sub(r"[^\w\s]", "", text)              # eliminar puntuación y caracteres especiales
    text = re.sub(r"\d+", "", text)                    # eliminar números sueltos
    return " ".join(text.split())                       # colapsar espacios múltiples

clean_udf = F.udf(clean_text, StringType())

df_cleaned = (df_raw
    .dropDuplicates(["id"])
    .withColumn("fecha_dt",     F.to_timestamp("fecha"))
    .withColumn("likes",        F.coalesce(F.col("likes"),    F.lit(0)))
    .withColumn("retweets",     F.coalesce(F.col("retweets"), F.lit(0)))
    .withColumn("texto_limpio", clean_udf(F.col("comentario")))
    # Filtrar tweets con texto vacío después de la limpieza
    .filter(F.length(F.col("texto_limpio")) > 5)
)

total_limpio = df_cleaned.count()
print(f"Registros después de limpieza: {total_limpio}")
print("\nEjemplo de texto original vs limpio:")
df_cleaned.select("comentario", "texto_limpio").limit(3).toPandas()

Registros después de limpieza: 5132

Ejemplo de texto original vs limpio:


,comentario,texto_limpio
0,Delegación de productores de papa nativa del s...,delegación de productores de papa nativa del s...
1,Precio de sol estable y Bolsa de Valores de Li...,precio de sol estable y bolsa de valores de li...
2,Precio del balón de gas no disminuye en distri...,precio del balón de gas no disminuye en distri...


In [11]:
# Análisis de sentimiento con BERT (pysentimiento)

from pyspark.sql.functions import pandas_udf
import pandas as pd

# El analyzer se carga una sola vez por proceso ejecutor (lazy singleton)
_analyzer = None

def get_analyzer():
    global _analyzer
    if _analyzer is None:
        from pysentimiento import create_analyzer
        _analyzer = create_analyzer(task="sentiment", lang="es")
    return _analyzer

@pandas_udf(FloatType())
def sentiment_score_udf(texts: pd.Series) -> pd.Series:
    """
    Aplica el modelo BERT de pysentimiento a un batch de textos.
    Retorna un float: 1.0 (positivo), 0.0 (neutro), -1.0 (negativo).
    Textos vacíos o con error retornan 0.0 (neutro).
    """
    model = get_analyzer()
    score_map = {"POS": 1.0, "NEU": 0.0, "NEG": -1.0}

    def predict_one(text):
        if not text or len(str(text).strip()) < 3:
            return 0.0
        try:
            return float(score_map.get(model.predict(str(text)).output, 0.0))
        except Exception:
            return 0.0

    return texts.apply(predict_one)

print("Iniciando análisis de sentimiento con BERT (pysentimiento)...")
print("Esto puede tomar varios minutos")

df_sentimiento = (df_cleaned
    .withColumn("sentiment_score", sentiment_score_udf(F.col("texto_limpio")))
    .withColumn("sentimiento",
        F.when(F.col("sentiment_score") >  0, "positivo")
         .when(F.col("sentiment_score") <  0, "negativo")
         .otherwise("neutro"))
)

print("Análisis completado.")

# Distribución de sentimientos
print("\nDistribución de sentimientos:")
df_sentimiento.groupBy("sentimiento").count().orderBy("sentimiento").show()

Iniciando análisis de sentimiento con BERT (pysentimiento)...
Esto puede tomar varios minutos
Análisis completado.

Distribución de sentimientos:
+-----------+-----+
|sentimiento|count|
+-----------+-----+
|   negativo| 2239|
|     neutro| 2458|
|   positivo|  435|
+-----------+-----+



In [12]:
# Agregar por semana

df_con_semana = df_sentimiento.withColumn(
    "semana", F.date_trunc("week", F.col("fecha_dt"))
)

df_semanal = (df_con_semana
    .groupBy("semana")
    .agg(
        F.count("id").alias("n_tweets"),
        F.avg("sentiment_score").alias("sentiment_promedio"),
        F.sum(F.when(F.col("sentimiento") == "negativo", 1).otherwise(0)).alias("n_negativos"),
        F.sum(F.when(F.col("sentimiento") == "positivo", 1).otherwise(0)).alias("n_positivos"),
        F.sum(F.when(F.col("sentimiento") == "neutro",   1).otherwise(0)).alias("n_neutros"),
        # Suma ponderada: tweets con más retweets tienen más peso en el sentimiento
        F.sum(F.col("retweets").cast("long")).alias("total_retweets"),
    )
    .withColumn("pct_negativo", F.round(F.col("n_negativos") / F.col("n_tweets") * 100, 2))
    .withColumn("pct_positivo", F.round(F.col("n_positivos") / F.col("n_tweets") * 100, 2))
    .orderBy("semana")
)

print(f"Semanas con datos: {df_semanal.count()}")
print("\nMuestra de métricas semanales:")
df_semanal.limit(5).toPandas()

Semanas con datos: 465

Muestra de métricas semanales:


,semana,n_tweets,sentiment_promedio,n_negativos,n_positivos,n_neutros,total_retweets,pct_negativo,pct_positivo
0,2015-01-05,1,1.000000,0,1,0,2,0.00,100.0
1,2015-01-12,2,-0.500000,1,0,1,6,50.00,0.0
2,2015-01-19,3,-0.666667,2,0,1,20,66.67,0.0
3,2015-01-26,2,-0.500000,1,0,1,6,50.00,0.0
4,2015-02-02,1,-1.000000,1,0,0,1,100.00,0.0


In [13]:
# Guardar en Zona Plata (GCS) como Parquet

# Seleccionar columnas finales de la tabla de tweets individuales
df_final_tweets = df_con_semana.select(
    "id", "fecha_dt", "semana",
    "comentario", "texto_limpio",
    "likes", "retweets",
    "termino_busqueda",
    "sentiment_score", "sentimiento"
)

# Guardar tabla de tweets individuales
df_final_tweets.write.mode("overwrite").parquet(PLATA_TWEETS)
print(f"Tabla tweets guardada en: {PLATA_TWEETS}")

# Guardar tabla de métricas semanales
df_semanal.write.mode("overwrite").parquet(PLATA_SEMANAL)
print(f"Tabla semanal guardada en: {PLATA_SEMANAL}")

print("\n" + "─" * 50)
print("Pipeline ETL completado exitosamente (Granularidad Semanal).")
print(f"  Tweets procesados : {df_final_tweets.count()}")
print(f"  Semanas generadas    : {df_semanal.count()}")
print("─" * 50)

Tabla tweets guardada en: gs://food-price-peru-2025-01/plata/tweets_sentimiento
Tabla semanal guardada en: gs://food-price-peru-2025-01/plata/tweets_sentimiento_semanal

──────────────────────────────────────────────────
Pipeline ETL completado exitosamente (Granularidad Semanal).
  Tweets procesados : 5132
  Semanas generadas    : 465
──────────────────────────────────────────────────


In [14]:
# Verificación final y resumen de calidad

print("=" * 50)
print("RESUMEN DE CALIDAD — ENTREGABLE 3 (SEMANAL)")
print("=" * 50)

# Verificar que los Parquet se pueden leer de vuelta
df_check_tweets = spark.read.parquet(PLATA_TWEETS)
df_check_semanal = spark.read.parquet(PLATA_SEMANAL)

print(f"\nTabla tweets_sentimiento:")
print(f"  Filas    : {df_check_tweets.count()}")
print(f"  Columnas : {df_check_tweets.columns}")

print(f"\nTabla tweets_sentimiento_semanal:")
print(f"  Filas    : {df_check_semanal.count()}")
print(f"  Columnas : {df_check_semanal.columns}")

print("\nDistribución de sentimientos final:")
(df_check_tweets.groupBy("sentimiento").count()
    .withColumn("pct", F.round(F.col("count") / df_check_tweets.count() * 100, 1))
    .orderBy("sentimiento").show())

print("\nRango temporal cubierto:")
df_check_tweets.agg(
    F.min("fecha_dt").alias("fecha_min"),
    F.max("fecha_dt").alias("fecha_max")
).show()

print("\nNulos en columnas críticas:")
for col in ["id", "texto_limpio", "sentiment_score", "sentimiento", "semana"]:
    nulos = df_check_tweets.filter(F.col(col).isNull()).count()
    print(f"  {col}: {nulos} nulos")

RESUMEN DE CALIDAD — ENTREGABLE 3 (SEMANAL)

Tabla tweets_sentimiento:
  Filas    : 5132
  Columnas : ['id', 'fecha_dt', 'semana', 'comentario', 'texto_limpio', 'likes', 'retweets', 'termino_busqueda', 'sentiment_score', 'sentimiento']

Tabla tweets_sentimiento_semanal:
  Filas    : 465
  Columnas : ['semana', 'n_tweets', 'sentiment_promedio', 'n_negativos', 'n_positivos', 'n_neutros', 'total_retweets', 'pct_negativo', 'pct_positivo']

Distribución de sentimientos final:
+-----------+-----+----+
|sentimiento|count| pct|
+-----------+-----+----+
|   negativo| 2239|43.6|
|     neutro| 2458|47.9|
|   positivo|  435| 8.5|
+-----------+-----+----+


Rango temporal cubierto:
+-------------------+-------------------+
|          fecha_min|          fecha_max|
+-------------------+-------------------+
|2015-01-10 22:48:22|2024-06-26 19:10:19|
+-------------------+-------------------+


Nulos en columnas críticas:
  id: 0 nulos
  texto_limpio: 0 nulos
  sentiment_score: 0 nulos
  sentimiento: 0 